# Assignment 3: Hybrid Semantic Retrieval & Intelligence System (HSRIS)
### NLP Pipeline | Customer Support Tickets | PyTorch from Scratch

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

True
2
Tesla T4


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/waseemalastal/customer-support-ticket-dataset/customer_support_tickets.csv
/kaggle/input/datasets/rtatman/glove-global-vectors-for-word-representation/glove.6B.200d.txt
/kaggle/input/datasets/rtatman/glove-global-vectors-for-word-representation/glove.6B.50d.txt
/kaggle/input/datasets/rtatman/glove-global-vectors-for-word-representation/glove.6B.100d.txt


In [4]:
import pandas as pd
df = pd.read_csv("/kaggle/input/datasets/waseemalastal/customer-support-ticket-dataset/customer_support_tickets.csv")
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [5]:
df.columns

Index(['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age',
       'Customer Gender', 'Product Purchased', 'Date of Purchase',
       'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status',
       'Resolution', 'Ticket Priority', 'Ticket Channel',
       'First Response Time', 'Time to Resolution',
       'Customer Satisfaction Rating'],
      dtype='object')

In [6]:
df.shape

(8469, 17)

## Part 1: Categorical Foundation — The Encoders

### 1a. Label Encoding — Ticket Priority

In [7]:
# Ordinal label encoding for Ticket Priority (from scratch — no sklearn)
priority_map = {"Low": 0, "Medium": 1, "High": 2, "Critical": 3}
df["priority_encoded"] = df["Ticket Priority"].map(priority_map)
# Handle any unseen / NaN priorities gracefully
df["priority_encoded"] = df["priority_encoded"].fillna(-1).astype(int)
df[["Ticket Priority", "priority_encoded"]].head(10)

,Ticket Priority,priority_encoded
0,Critical,3
1,Critical,3
2,Low,0
3,Low,0
4,Low,0
5,Low,0
6,Critical,3
7,Critical,3
8,Low,0
9,Critical,3


### 1b. One-Hot Encoding — Ticket Channel

In [8]:
# One-Hot Encoding for Ticket Channel (from scratch — no sklearn)
known_channels = sorted(df["Ticket Channel"].dropna().unique().tolist())

def one_hot_channel(value, categories):
    """Return a binary vector; unknown category -> all-zero vector."""
    vec = [0] * len(categories)
    if value in categories:
        vec[categories.index(value)] = 1
    return vec

for ch in known_channels:
    col_name = "channel_" + ch.replace(" ", "_")
    df[col_name] = df["Ticket Channel"].apply(lambda x: 1 if x == ch else 0)

channel_cols = ["channel_" + ch.replace(" ", "_") for ch in known_channels]
print("Channels encoded:", known_channels)
df[["Ticket Channel"] + channel_cols].head(8)

Channels encoded: ['Chat', 'Email', 'Phone', 'Social media']


,Ticket Channel,channel_Chat,channel_Email,channel_Phone,channel_Social_media
0,Social media,0,0,0,1
1,Chat,1,0,0,0
2,Social media,0,0,0,1
3,Social media,0,0,0,1
4,Email,0,1,0,0
5,Social media,0,0,0,1
6,Social media,0,0,0,1
7,Social media,0,0,0,1


In [9]:
# Verify unseen category handling
test_unseen = one_hot_channel("Fax", known_channels)
print("Unseen channel Fax ->", test_unseen)  # should be all zeros

Unseen channel Fax -> [0, 0, 0, 0]


## Part 2: Sparse Representation — Keyword Retrieval

### 2a. Preprocessing & Display Text Construction

In [10]:
import re

def build_display_text(row):
    subject     = str(row.get("Ticket Subject", "")).strip()
    description = str(row.get("Ticket Description", ""))
    product     = str(row.get("Product Purchased", "the product")).strip()

    # 1. Replace template placeholder with actual product name
    description = description.replace("{product_purchased}", product)

    # 2. Remove ALL-CAPS tokens early (INFO, ERROR, WARN, PM ...)
    description = re.sub(r'\b[A-Z]{2,}\b', ' ', description)

    # 3. Remove log/thread fragments anywhere in string
    description = re.sub(
        r'(pm\s+)?client\s+thread\s+.*?(?=i\'ve|i\'m|my\s|the\s|we\s|$)',
        ' ', description, flags=re.IGNORECASE)

    # 4. Remove snake_case and PascalCase code tokens
    description = re.sub(r'\b[a-zA-Z]+(?:_[a-zA-Z]+)+\b', ' ', description)
    description = re.sub(r'\b[A-Z][a-z]+(?:[A-Z][a-z0-9]+)+\b', ' ', description)

    # 5. Remove timestamps e.g. 2023-06-01 12:15:36
    description = re.sub(r'\b\d{1,4}[/-]\d{1,2}[/-]\d{1,4}(\s+\d{1,2}:\d{2}(:\d{2})?)?\b', ' ', description)

    # 6. Remove standalone times e.g. 17:57:11
    description = re.sub(r'\b\d{1,2}:\d{2}(:\d{2})?\b', ' ', description)

    # 7. Remove IP addresses
    description = re.sub(r'\b\d{1,3}(\.\d{1,3}){3}\b', ' ', description)

    # 8. Remove currency / prices e.g. $1,000.00
    description = re.sub(r'\$[\d,]+(\.\d+)?', ' ', description)

    # 9. Remove dimension patterns e.g. 5 1/2
    description = re.sub(r'\d+\s*\d*/\d+', ' ', description)

    # 10. Remove remaining standalone numbers
    description = re.sub(r'\b\d+(\.\d+)?\b', ' ', description)

    # 11. Remove URLs
    description = re.sub(r'http\S+|www\.\S+', ' ', description)

    # 12. Remove bullet / list markers
    description = re.sub(r'[*\u2022\-\u2013\u2014|]', ' ', description)

    # 13. Remove non-alphanumeric except basic punctuation
    description = re.sub(r'[^\w\s.,!?\']', ' ', description)

    # 14. Collapse whitespace
    description = re.sub(r'\s+', ' ', description).strip()

    # 15. Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', description)

    # 16. Filter out generic / broken / noisy sentences
    generic_patterns = [
        r"(?i)^i'?m having an issue with the .{0,60}$",
        r"(?i)^i'?m having an issue with the .{0,40}please assist",
        r"(?i)^i'?m facing a problem with my .{0,60}$",
        r"(?i)^please assist\.?$",
        r"(?i)^i need assistance as soon as possible",
        r"(?i)^if my product failed to reach",
        r"(?i)^i have a product .{0,40}please assist",
        r"(?i)^failed to \w+",
        r"(?i)^we will (not )?make any money",
        r"(?i)^if such payment is not possible",
        r"(?i)^try this",
        r"(?i)^client\s+thread",
        r"(?i)^to \w+ (package|value|store)",
    ]

    def is_meaningful(s):
        s = s.strip()
        if len(s) < 25:
            return False
        for pat in generic_patterns:
            if re.match(pat, s):
                return False
        return True

    good = [s.strip() for s in sentences if is_meaningful(s)]
    short_desc = " ".join(good[:2]) if good else description[:200]
    return subject, short_desc

df["display_subject"], df["display_description"] = zip(*df.apply(build_display_text, axis=1))
df["text"] = df["display_subject"] + " " + df["display_description"]

print("Sample display text:")
print(df[["display_subject", "display_description"]].head(5).to_string())

Sample display text:
            display_subject                                                                                                                                                                                                                        display_description
0             Product setup                                                                                                                                                        Your billing zip code is . We appreciate that you have requested a website address.
1  Peripheral compatibility                                                                                                                                                        If you need to change an existing product. If The issue I'm facing is intermittent.
2           Network problem                                                                                                                                               The Dell is not turn

### 2b. Text Cleaning & Custom Tokenizer

In [11]:
def clean_text(t):
    """Clean text for tokenization / TF-IDF / embedding (no sklearn)."""
    t = str(t)

    # Remove template placeholders
    t = re.sub(r'\{[^}]*\}', ' ', t)

    # Remove ALL-CAPS tokens
    t = re.sub(r'\b[A-Z]{2,}\b', ' ', t)

    # Remove log/thread fragments
    t = re.sub(
        r'(pm\s+)?client\s+thread\s+.*?(?=i\'ve|i\'m|my\s|the\s|we\s|$)',
        ' ', t, flags=re.IGNORECASE)

    # Remove snake_case and PascalCase code tokens
    t = re.sub(r'\b[a-zA-Z]+(?:_[a-zA-Z]+)+\b', ' ', t)
    t = re.sub(r'\b[A-Z][a-z]+(?:[A-Z][a-z0-9]+)+\b', ' ', t)

    # Remove timestamps, IPs, prices, numbers
    t = re.sub(r'\b\d{1,4}[/-]\d{1,2}[/-]\d{1,4}(\s+\d{1,2}:\d{2}(:\d{2})?)?\b', ' ', t)
    t = re.sub(r'\b\d{1,2}:\d{2}(:\d{2})?\b', ' ', t)
    t = re.sub(r'\b\d{1,3}(\.\d{1,3}){3}\b', ' ', t)
    t = re.sub(r'\$[\d,]+(\.\d+)?', ' ', t)
    t = re.sub(r'\d+\s*\d*/\d+', ' ', t)
    t = re.sub(r'\b\d+(\.\d+)?\b', ' ', t)

    # Remove URLs, bullets, special chars
    t = re.sub(r'http\S+|www\.\S+', ' ', t)
    t = re.sub(r'[*\u2022\-\u2013\u2014|\\]', ' ', t)
    t = re.sub(r'[^a-zA-Z\s]', ' ', t)

    # Remove very short tokens (1-2 chars)
    t = re.sub(r'\b[a-zA-Z]{1,2}\b', ' ', t)

    # Lowercase and collapse whitespace
    t = re.sub(r'\s+', ' ', t).strip().lower()
    return t

df["clean_text"] = df["text"].apply(clean_text)
df[["text", "clean_text"]].head(3)

,text,clean_text
0,Product setup Your billing zip code is . We ap...,product setup your billing zip code appreciate...
1,Peripheral compatibility If you need to change...,peripheral compatibility you need change exist...
2,Network problem The Dell is not turning on. It...,network problem the dell not turning was worki...


In [12]:
# Custom regex-based tokenizer (no sklearn)
def tokenize(text):
    return re.findall(r'\b[a-z]+\b', text)

df["tokens"] = df["clean_text"].apply(tokenize)
df["tokens"].head()

0    [product, setup, your, billing, zip, code, app...
1    [peripheral, compatibility, you, need, change,...
2    [network, problem, the, dell, not, turning, wa...
3    [account, access, you, have, problem, you, int...
4    [data, loss, note, the, seller, not, responsib...
Name: tokens, dtype: object

### 2c. Count Vectorizer (Bag of Words) — Top 5,000 Tokens

In [13]:
from collections import Counter
import numpy as np

word_counter = Counter()
for tokens in df["tokens"]:
    word_counter.update(tokens)

print("Total unique words:", len(word_counter))

vocab_size = 5000
vocab    = [w for w, _ in word_counter.most_common(vocab_size)]
word2idx = {w: i for i, w in enumerate(vocab)}

print("Vocabulary size:", len(vocab))
print("Sample vocab:", vocab[:10])

Total unique words: 4686
Vocabulary size: 4686
Sample vocab: ['the', 'issue', 'and', 'product', 'you', 'this', 'but', 'problem', 'for', 'not']


In [14]:
def bow_vector(tokens):
    """Bag-of-words count vector (from scratch)."""
    vec = np.zeros(vocab_size)
    for w in tokens:
        if w in word2idx:
            vec[word2idx[w]] += 1
    return vec

df["bow"] = df["tokens"].apply(bow_vector)
print("BoW vector shape:", df["bow"].iloc[0].shape)
df["bow"].iloc[0][:20]

BoW vector shape: (5000,)


array([0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 1., 0.])